# Friend A — exp38: public exp29 + per-video AdaBN

This notebook adapts BatchNorm statistics for the public **primary pilkwang** and **secondary seed314159** separately on each unlabelled video, then runs the unchanged exp29 stack. The label-free guard accepts AdaBN only when `N_adapted / N_base` is inside `[0.90, 1.05]`; otherwise that video's untouched exp29 graph is used.

The fixed 12-video 44b6 run is a regression and timing check only: the public weights saw these videos. It is not an honest validation and this notebook never submits to the leaderboard.

## Stop and resume

- You may use **Interrupt Kernel**. A currently running public-inference video may need to restart; completed videos are retained.
- For a cooperative stop, create `C:\Projects\CellTracker\outputs\local_friend_a_exp05_public_adabn\STOP_REQUESTED`.
- Re-run all cells to resume. Set `BIOHUB_EXP38_MAX_VIDEOS=1` before Jupyter to process one unfinished video at a time.
- The existing B20 exp29 cache is detected automatically, so the baseline normally does not use GPU time.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, start / '.github-push-worktree']
for parent in start.parents:
    candidates.extend([parent, parent / '.github-push-worktree'])
REPO_ROOT = next((p for p in candidates if (p / 'TEAM.md').is_file() and (p / 'tools' / 'public_local.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Repository root not found. Start Jupyter from C:\\Projects\\CellTracker.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from experiments.local_friend_a_exp05_public_adabn.runner import (
    Exp38Config, clear_stop, compact_status, preflight, run_pipeline, stop_path,
)
print('Repository:', REPO_ROOT)


In [ ]:
# Settings may be overridden with BIOHUB_EXP38_* before Jupyter starts.
cfg = Exp38Config.from_environment(REPO_ROOT)
cfg.serialisable()


In [ ]:
# Clears only a stale stop request. It does not remove results or checkpoints.
clear_stop(cfg)
check = preflight(cfg, gpu_smoke=cfg.run_pipeline)
check


The next cell performs the resumable run. It first reuses or creates the exp29 control graph for a video, recalibrates both public models, calculates the adapted graph, and records the guard decision. GT is opened only after all 12 prediction sets exist, during official scoring.


In [ ]:
state = run_pipeline(cfg)
compact_status(cfg)


In [ ]:
status = compact_status(cfg)
print(status)
result_path = cfg.output_dir / 'result.md'
if result_path.is_file():
    print(result_path.read_text(encoding='utf-8'))
print('External stop marker:', stop_path(cfg))
print('Kaggle cell:', REPO_ROOT / 'experiments/local_friend_a_exp05_public_adabn/kaggle_adabn_cell.py')
